In [6]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=rXi6KYrRRBMR39VLwsCH8NVtrxL8vE&access_type=offline&code_challenge=5vM_vzHRUTG3qP1OxTTXseOMh-vJCoiHIo_6NYZnJaQ&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning

In [1]:
import numpy as np
from typing import Any
import os
import hail as hl
import pyspark.sql.functions as f
import pandas as pd

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.l2g_prediction import L2GPrediction
hail_dir = os.path.dirname(hl.__file__)
session = Session(hail_home=hail_dir, start_hail=True, extended_spark_conf={"spark.driver.memory": "12g","spark.kryoserializer.buffer.max": "500m","spark.driver.maxResultSize":"2g"})
hl.init(sc=session.spark.sparkContext, log="/dev/null")

Loading BokehJS ...

24/04/10 18:06:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://mib118093s.internal.sanger.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null


In [2]:
from pyspark.sql.types import *

schema = StructType([
    StructField("datasourceId", StringType(), True),
    StructField("targetId", StringType(), True),
    StructField("alleleOrigins", ArrayType(StringType()), True),
    StructField("allelicRequirements", ArrayType(StringType()), True),
    StructField("ancestry", StringType(), True),
    StructField("ancestryId", StringType(), True),
    StructField("beta", DoubleType(), True),
    StructField("betaConfidenceIntervalLower", DoubleType(), True),
    StructField("betaConfidenceIntervalUpper", DoubleType(), True),
    StructField("biologicalModelAllelicComposition", StringType(), True),
    StructField("biologicalModelGeneticBackground", StringType(), True),
    StructField("biologicalModelId", StringType(), True),
    StructField("biomarkerList", ArrayType(StructType([
        StructField("description", StringType(), True),
        StructField("name", StringType(), True)
    ])), True),
    StructField("biomarkerName", StringType(), True),
    StructField("biomarkers", StructType([
        StructField("geneExpression", ArrayType(StructType([
            StructField("id", StringType(), True),
            StructField("name", StringType(), True)
        ])), True),
        StructField("variant", ArrayType(StructType([
            StructField("functionalConsequenceId", StringType(), True),
            StructField("id", StringType(), True),
            StructField("name", StringType(), True)
        ])), True)
    ]), True),
    StructField("biosamplesFromSource", ArrayType(StringType()), True),
    StructField("cellLineBackground", StringType(), True),
    StructField("cellType", StringType(), True),
    StructField("clinicalPhase", DoubleType(), True),
    StructField("clinicalSignificances", ArrayType(StringType()), True),
    StructField("clinicalStatus", StringType(), True),
    StructField("cohortDescription", StringType(), True),
    StructField("cohortId", StringType(), True),
    StructField("cohortPhenotypes", ArrayType(StringType()), True),
    StructField("cohortShortName", StringType(), True),
    StructField("confidence", StringType(), True),
    StructField("contrast", StringType(), True),
    StructField("crisprScreenLibrary", StringType(), True),
    StructField("datatypeId", StringType(), True),
    StructField("diseaseCellLines", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("tissue", StringType(), True),
        StructField("tissueId", StringType(), True)
    ])), True),
    StructField("diseaseFromSource", StringType(), True),
    StructField("diseaseFromSourceId", StringType(), True),
    StructField("diseaseFromSourceMappedId", StringType(), True),
    StructField("diseaseModelAssociatedHumanPhenotypes", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("label", StringType(), True)
    ])), True),
    StructField("diseaseModelAssociatedModelPhenotypes", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("label", StringType(), True)
    ])), True),
    StructField("drugFromSource", StringType(), True),
    StructField("drugId", StringType(), True),
    StructField("drugResponse", StringType(), True),
    StructField("expectedConfidence", StringType(), True),
    StructField("geneInteractionType", StringType(), True),
    StructField("geneticBackground", StringType(), True),
    StructField("geneticInteractionPValue", DoubleType(), True),
    StructField("geneticInteractionScore", DoubleType(), True),
    StructField("interactingTargetFromSourceId", StringType(), True),
    StructField("interactingTargetRole", StringType(), True),
    StructField("literature", ArrayType(StringType()), True),
    StructField("log2FoldChangePercentileRank", LongType(), True),
    StructField("log2FoldChangeValue", DoubleType(), True),
    StructField("mutatedSamples", ArrayType(StructType([
        StructField("functionalConsequenceId", StringType(), True),
        StructField("numberMutatedSamples", DoubleType(), True),
        StructField("numberSamplesTested", DoubleType(), True),
        StructField("numberSamplesWithMutationType", LongType(), True)
    ])), True),
    StructField("oddsRatio", DoubleType(), True),
    StructField("oddsRatioConfidenceIntervalLower", DoubleType(), True),
    StructField("oddsRatioConfidenceIntervalUpper", DoubleType(), True),
    StructField("pValueExponent", LongType(), True),
    StructField("pValueMantissa", DoubleType(), True),
    StructField("pathways", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("name", StringType(), True)
    ])), True),
    StructField("phenotypicConsequenceFDR", DoubleType(), True),
    StructField("phenotypicConsequenceLogFoldChange", DoubleType(), True),
    StructField("phenotypicConsequencePValue", DoubleType(), True),
    StructField("pmcIds", ArrayType(StringType()), True),
    StructField("projectDescription", StringType(), True),
    StructField("projectId", StringType(), True),
    StructField("publicationFirstAuthor", StringType(), True),
    StructField("publicationYear", LongType(), True),
    StructField("reactionId", StringType(), True),
    StructField("reactionName", StringType(), True),
    StructField("releaseDate", StringType(), True),
    StructField("releaseVersion", StringType(), True),
    StructField("resourceScore", DoubleType(), True),
    StructField("sex", ArrayType(StringType()), True),
    StructField("significantDriverMethods", ArrayType(StringType()), True),
    StructField("statisticalMethod", StringType(), True),
    StructField("statisticalMethodOverview", StringType(), True),
    StructField("statisticalTestTail", StringType(), True),
    StructField("studyCases", LongType(), True),
    StructField("studyCasesWithQualifyingVariants", LongType(), True),
    StructField("studyId", StringType(), True),
    StructField("studyOverview", StringType(), True),
    StructField("studySampleSize", LongType(), True),
    StructField("studyStartDate", StringType(), True),
    StructField("studyStopReason", StringType(), True),
    StructField("studyStopReasonCategories", ArrayType(StringType()), True),
    StructField("targetFromSource", StringType(), True),
    StructField("targetFromSourceId", StringType(), True),
    StructField("targetInModel", StringType(), True),
    StructField("targetInModelEnsemblId", StringType(), True),
    StructField("targetInModelMgiId", StringType(), True),
    StructField("targetModulation", StringType(), True),
    StructField("targetRole", StringType(), True),
    StructField("textMiningSentences", ArrayType(StructType([
        StructField("dEnd", LongType(), True),
        StructField("dStart", LongType(), True),
        StructField("section", StringType(), True),
        StructField("tEnd", LongType(), True),
        StructField("tStart", LongType(), True),
        StructField("text", StringType(), True)
    ])), True),
    StructField("urls", ArrayType(StructType([
        StructField("niceName", StringType(), True),
        StructField("url", StringType(), True)
    ])), True),
    StructField("validationHypotheses", ArrayType(StructType([
        StructField("description", StringType(), True),
        StructField("name", StringType(), True),
        StructField("status", StringType(), True)
    ])), True),
    StructField("variantAminoacidDescriptions", ArrayType(StringType()), True),
    StructField("variantFunctionalConsequenceFromQtlId", StringType(), True),
    StructField("variantFunctionalConsequenceId", StringType(), True),
    StructField("variantHgvsId", StringType(), True),
    StructField("variantId", StringType(), True),
    StructField("variantRsId", StringType(), True),
    StructField("sourceId", StringType(), True),
    StructField("diseaseId", StringType(), True),
    StructField("id", StringType(), True),
    StructField("score", DoubleType(), True),
    StructField("variantEffect", StringType(), True),
    StructField("directionOnTrait", StringType(), True)
])


In [5]:

path="gs://open-targets-data-releases/24.03/output/etl/parquet/evidence"
path="/Users/yt4/Projects/ot_data/evidence"

df=session.read_parquet(path,schema=schema)

In [6]:
df.count()

17317290

In [7]:
df.show(2)

+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+-------------+----------+--------------------+------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+-----------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+------+------------+------------------+-------------------+-----------------+------------------------+-----------------------+-----------------------------+---------------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+----